# 03. 파인튜닝 전후 비교 데모

저장된 LoRA 어댑터를 불러와, 같은 질문에 대해 베이스 모델(어댑터 비활성화)과 DPO 모델의 응답을 비교합니다.
`adapter_path`를 본인 Drive의 어댑터 경로로 바꿔서 실행하세요.

> 샘플링(`do_sample=True`)으로 생성하므로 실행할 때마다 결과가 달라집니다.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes

from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen25-7b-dpo-fluff-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print("모델 로드 완료")

In [ ]:
def generate_response(prompt_text, use_adapter=True, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        if use_adapter:
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        else:
            with model.disable_adapter():
                output = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
test_prompt = """I am an elderly senior citizen over 65 years old with very low digital literacy. I got a text message saying my package delivery failed and I need to click a link to pay a small fee to get it redelivered. I don't remember ordering anything. What steps should I take right now to secure my accounts?"""

print("=== 파인튜닝 전 (베이스 모델) ===")
base_response = generate_response(test_prompt, use_adapter=False)
print(base_response)

print("\n=== 파인튜닝 후 (DPO 적용) ===")
finetuned_response = generate_response(test_prompt, use_adapter=True)
print(finetuned_response)

print(f"\n[길이 비교] 베이스: {len(base_response)}자 / 파인튜닝: {len(finetuned_response)}자")